In [11]:
from app.core import settings

In [12]:
api_key = settings.OPENAI_API_KEY

In [13]:
# import bs4
# from langchain import hub
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain_community.document_loaders import WebBaseLoader
# from langchain_community.vectorstores import Chroma
# from langchain_core.output_parsers import StrOutputParser
# from langchain_core.runnables import RunnablePassthrough
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# #### INDEXING ####

# # Load Documents
# loader = WebBaseLoader(
#     web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
#     bs_kwargs=dict(
#         parse_only=bs4.SoupStrainer(
#             class_=("post-content", "post-title", "post-header")
#         )
#     ),
# )
# docs = loader.load()

# # Split
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# splits = text_splitter.split_documents(docs)

# # Embed
# vectorstore = Chroma.from_documents(documents=splits, 
#                                     embedding=OpenAIEmbeddings())

# retriever = vectorstore.as_retriever()

# #### RETRIEVAL and GENERATION ####

# # Prompt
# prompt = hub.pull("rlm/rag-prompt")

# # LLM
# llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# # Post-processing
# def format_docs(docs):
#     return "\n\n".join(doc.page_content for doc in docs)

# # Chain
# rag_chain = (
#     {"context": retriever | format_docs, "question": RunnablePassthrough()}
#     | prompt
#     | llm
#     | StrOutputParser()
# )

# # Question
# rag_chain.invoke("What is Task Decomposition?")

In [ ]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_tokens_from_string(question, "cl100k_base")

In [ ]:
from langchain_openai import OpenAIEmbeddings
embd = OpenAIEmbeddings()
query_result = embd.embed_query(question)
document_result = embd.embed_query(document)
len(query_result)

In [ ]:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)

In [352]:
import re
from langchain_community.document_loaders import PyMuPDFLoader


# همه کاراکترهای فارسی + اعداد فارسی + اعداد انگلیسی
FA = r'[\u0600-\u06FF\uFB50-\uFDFF\uFE70-\uFEFF\u06F0-\u06F90-9]'
EN = r'[A-Za-z0-9]'

def fix_persian_text2(text: str) -> str:
    # اتصال کاراکترهای انگلیسی که با \n جدا شدن
    text = re.sub(rf'(?<={EN})[ \t]*\n[ \t]*(?={EN})', '', text)
    
    # اتصال انگلیسی به فارسی/عددی
    text = re.sub(rf'(?<={EN})[ \t]*\n[ \t]*(?={FA})', ' ', text)
    
    # اتصال فارسی/عددی به انگلیسی
    text = re.sub(rf'(?<={FA})[ \t]*\n[ \t]*(?={EN})', ' ', text)

    # اتصال اجزای فارسی/عددی که با \n جدا شدن (نیم‌فاصله)
    text = re.sub(rf'(?<={FA})\n(?={FA})', '\u200c', text)
    
    # چند خط خالی پشت سر هم رو به یک خط خالی
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()
loader = PyMuPDFLoader("docs/rag_docs.pdf")
docs = loader.load()

for doc in docs:
    doc.page_content = fix_persian_text2(doc.page_content)

In [ ]:
# from langchain_community.document_loaders import PyPDFLoader

# loader = PyPDFLoader("docs/rag_docs.pdf")
# docs = loader.load()

In [345]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("docs/rag_docs.pdf")
docs = loader.load()

In [228]:
docs[3]

Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-06-24T18:09:36+03:30', 'source': 'docs/rag_docs.pdf', 'file_path': 'docs/rag_docs.pdf', 'total_pages': 15, 'format': 'PDF 1.7', 'title': '', 'author': 'hamed safaei', 'subject': '', 'keywords': '', 'moddate': '2026-06-24T18:09:36+03:30', 'trapped': '', 'modDate': "D:20260624180936+03'30'", 'creationDate': "D:20260624180936+03'30'", 'page': 4}, page_content='۲.۴\n پایگاه داده برداری (Vector Store)\n \nالگوریتم\u200cهای جستجو در پایگاه های داده برداری معموالً بر اساس معیارهایی مانند شباهت کسینوسی \n(Cosine Similarity) یا \n فاصله اقلیدسی (Euclidean Distance) عمل می\u200cکنند. برای مقیاس ،های بزرگ\u200cاز الگوریتم\u200cهایی نظیر HNSW (Hierarchical Navigable Small World) و IVF (Inverted File Index) \nاستفاده  می\u200cشود  که  جستجوی  تقریبی  نزدیک\u200cترین  همسایه\u200cها (ANN - Approximate NearestNeighbor) را با سرعت بسیار باال انجام می\u200cدهند.\n \n \n۲.۵\n بازیابی و رتبه\u2

In [353]:
for i, doc in enumerate(docs[:5]):
    print(f"Page {i+1}")
    print(doc.page_content[:2000])
    print("-" * 50)

Page 1
۱
 .
 مقدمه و پیشینه 
د‌ر دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه‌های اصلی نرم افزارهای هوشمند تبدیل‌شده‌اند. این مدل‌ها قادرند متون طبیعی را درک کرده، آن‌ها را تحلیل کنند و پاسخ هایی دقیق و روان
 تولید نمایند. با این حال، یکی از بزرگ‌ترین چالش‌های این مدل‌ها آن است که دانش آن ها به زمان‌آموزش محدود می‌شود و دسترسی مستقیمی به داده‌های جدید، اختصاصی یا سازمانی ندارند.
 
 
برای رفع این محدودیت، رویکرد «بازیابی-
 افزوده تولید» یاRAG (Retrieval-AugmentedGeneration) 
 مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:
 
1. در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می‌شود.
 
2. در مرحله دوم، مدل زبانی از این اطالعات بازیابی‌شده به عنوان زمینه (Context) استفاده می کند‌تا پاسخ نهایی را تولید نماید.
 
 
معماری RAG نخستین بار توسط‌لوئیس و همکارانش 
 در سال۲۰۲۰
 
 معرفی شد و از آن زمان تاکنون‌تحوالت گسترده ای را تجربه کرده است. امروزه این معماری در طیف وسیعی از کاربردها، از جمله‌پرسش و پاسخ سازمانی، چت‌بات‌های پشتیبانی مشتری، تحلیل اسناد حقوقی، سیستم های پزشکی

In [42]:
print(docs[0].metadata)

{'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-06-24T18:09:36+03:30', 'source': 'docs/rag_docs.pdf', 'file_path': 'docs/rag_docs.pdf', 'total_pages': 15, 'format': 'PDF 1.7', 'title': '', 'author': 'hamed safaei', 'subject': '', 'keywords': '', 'moddate': '2026-06-24T18:09:36+03:30', 'trapped': '', 'modDate': "D:20260624180936+03'30'", 'creationDate': "D:20260624180936+03'30'", 'page': 0}


In [83]:
print(repr(docs[0].page_content[:1000]))

'مقدمه و پیشینه\nدر دنیای امروز، مدل\u200cهای زبانی بزرگ (LLM) به یکی از پایه\nهای اصلی نرم افزارهای هوشمند تبدیل\nشده\nاند. این مدل\nها قادرند متون طبیعی را درک کرده، آن\nها را تحلیل کنند و پاسخ\u200cهایی دقیق و روان\nتولید نمایند. با این حال، یکی از بزرگ\nترین چالش\nهای این مدل\nها آن است که دانش آن\u200cها به زمان\nآموزش محدود می\nشود و دسترسی مستقیمی به داده\nهای جدید، اختصاصی یا سازمانی ندارند. \n \n \nبرای رفع این محدودیت، رویکرد «بازیابی-افزوده تولید» یا\nRAG (Retrieval-Augmented\nGeneration) \nمطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است: \n \n۱. در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می\nشود. \n \n۲. در مرحله دوم، مدل زبانی از این اطالعات بازیابی\nشده به عنوان زمینه (Context) استفاده می\u200cکند\nتا پاسخ نهایی را تولید نماید. \n \n \nمعماری RAG نخستین بار توسط لوئیس و همکارانش\nدر سال ۲۰۲۰\n \nمعرفی شد و از آن زمان تاکنون\nتحوالت گسترده\u200cای را تجربه کرده است. امروزه این معماری در طیف وسیعی از کاربردها، از جمله\nپرسش و پاسخ سازمانی، چت\nبات\nهای پشت

In [358]:
def extract_headings_merged(pdf_path: str, min_size: float = 18.0) -> list[dict]:
    doc = fitz.open(pdf_path)
    headings = []
    current = None
    
    for page_num, page in enumerate(doc, start=1):
        blocks = page.get_text("dict")["blocks"]
        
        for block in blocks:
            if block.get("type") != 0:
                continue
            
            for line in block.get("lines", []):
                line_text = ""
                line_size = 0
                
                for span in line.get("spans", []):
                    line_text += span["text"]
                    line_size = max(line_size, span["size"])
                
                line_text = line_text.strip()
                if not line_text:
                    continue
                
                if line_size >= min_size:
                    if current:
                        # اگر heading قبلی هنوز باز بود، merge کن
                        current["text"] += " " + line_text
                    else:
                        current = {"page": page_num, "text": line_text, "size": round(line_size, 1)}
                else:
                    if current:
                        headings.append(current)
                        current = None
    
    if current:
        headings.append(current)
    
    doc.close()
    return headings


headings = extract_headings_merged("docs/rag_docs.pdf", min_size=18.0)

for h in headings:
    print(f"{h['text']}")

۱ . مقدمه و پیشینه
۲ . اجزای اصلی یک سیستم RAG
۳ . چالش های RAG برای زبان فارسی
۴ . معیارهای ارزیابی RAG
۵ . معماری های پیشرفته RAG
۶ . پیاده سازی عملی با LangChain
۷ . کاربرد RAG در هوش تجاری (BI)
۸ . بهترین رویکردها و توصیه ها
۹ . آینده RAG
۱۰ . نتیجه گیری


In [342]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
loader = PyMuPDFLoader("docs/rag_docs.pdf")
pages = loader.load()

full_text = "\n".join([page.page_content for page in pages])

docs = Document(
    page_content=full_text,
    metadata={
        "source": "docs/rag_docs.pdf"
    }
)


In [343]:
for doc in docs:
    docs.page_content = fix_persian_text2(docs.page_content)

In [344]:
print(docs.page_content)

۱
 .
 مقدمه و پیشینه 
د‌ر دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه‌های اصلی نرم افزارهای هوشمند تبدیل‌شده‌اند. این مدل‌ها قادرند متون طبیعی را درک کرده، آن‌ها را تحلیل کنند و پاسخ هایی دقیق و روان
 تولید نمایند. با این حال، یکی از بزرگ‌ترین چالش‌های این مدل‌ها آن است که دانش آن ها به زمان‌آموزش محدود می‌شود و دسترسی مستقیمی به داده‌های جدید، اختصاصی یا سازمانی ندارند.
 
 
برای رفع این محدودیت، رویکرد «بازیابی-
 افزوده تولید» یاRAG (Retrieval-AugmentedGeneration) 
 مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:
 
1. در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می‌شود.
 
2. در مرحله دوم، مدل زبانی از این اطالعات بازیابی‌شده به عنوان زمینه (Context) استفاده می کند‌تا پاسخ نهایی را تولید نماید.
 
 
معماری RAG نخستین بار توسط‌لوئیس و همکارانش 
 در سال۲۰۲۰
 
 معرفی شد و از آن زمان تاکنون‌تحوالت گسترده ای را تجربه کرده است. امروزه این معماری در طیف وسیعی از کاربردها، از جمله‌پرسش و پاسخ سازمانی، چت‌بات‌های پشتیبانی مشتری، تحلیل اسناد حقوقی، سیستم های پزشکی و‌هوش 

In [304]:
docs

Document(metadata={}, page_content='مقدمه و پیشینه \nدر دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه\u200cهای اصلی نرم افزارهای هوشمند تبدیل\u200cشده\u200cاند. این مدل\u200cها قادرند متون طبیعی را درک کرده، آن\u200cها را تحلیل کنند و پاسخ هایی دقیق و روان\n تولید نمایند. با این حال، یکی از بزرگ\u200cترین چالش\u200cهای این مدل\u200cها آن است که دانش آن ها به زمان\u200cآموزش محدود می\u200cشود و دسترسی مستقیمی به داده\u200cهای جدید، اختصاصی یا سازمانی ندارند.\n \n \nبرای رفع این محدودیت، رویکرد «بازیابی-افزوده تولید» یا RAG (Retrieval-AugmentedGeneration) \n مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:\n \n1. در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می\u200cشود.\n \n2. در مرحله دوم، مدل زبانی از این اطالعات بازیابی\u200cشده به عنوان زمینه (Context) استفاده می کند\u200cتا پاسخ نهایی را تولید نماید.\n \n \nمعماری RAG نخستین بار توسط لوئیس و همکارانش \n در سال۲۰۲۰\n \n معرفی شد و از آن زمان تاکنون\u200cتحوالت گسترده ای را تجربه کرده است. امروزه این معماری در ط

In [305]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=[
        "\n\n",       # اولویت ۱: پاراگراف
        "\n",         # اولویت ۲: خط
        ".",          # اولویت ۳: پایان جمله (نقطه)
        "،",          # اولویت ۴: ویرگول فارسی
        " ",          # اولویت ۵: فاصله بین کلمات
        "\u200c",     # اولویت ۶: نیم‌فاصله
        ""            # اولویت آخر: کاراکتر
    ]
)

chunks = text_splitter.split_documents([docs])

In [306]:
for i, chunk in enumerate(chunks[:30]):
    print(i, len(chunk.page_content))

0 782
1 993
2 956
3 949
4 944
5 986
6 923
7 922
8 972
9 979
10 981
11 954
12 906
13 939
14 916
15 996
16 942
17 956
18 982
19 595


In [307]:
print("Chunks:", len(chunks))

for i in range(12):
    print(f"\nChunk {i}")
    print(chunks[i].page_content[:1300])
    print("-" * 50)

Chunks: 20

Chunk 0
مقدمه و پیشینه 
در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه‌های اصلی نرم افزارهای هوشمند تبدیل‌شده‌اند. این مدل‌ها قادرند متون طبیعی را درک کرده، آن‌ها را تحلیل کنند و پاسخ هایی دقیق و روان
 تولید نمایند. با این حال، یکی از بزرگ‌ترین چالش‌های این مدل‌ها آن است که دانش آن ها به زمان‌آموزش محدود می‌شود و دسترسی مستقیمی به داده‌های جدید، اختصاصی یا سازمانی ندارند.
 
 
برای رفع این محدودیت، رویکرد «بازیابی-افزوده تولید» یا RAG (Retrieval-AugmentedGeneration) 
 مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:
 
1. در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می‌شود.
 
2. در مرحله دوم، مدل زبانی از این اطالعات بازیابی‌شده به عنوان زمینه (Context) استفاده می کند‌تا پاسخ نهایی را تولید نماید.
 
 
معماری RAG نخستین بار توسط لوئیس و همکارانش 
 در سال۲۰۲۰
--------------------------------------------------

Chunk 1
2. در مرحله دوم، مدل زبانی از این اطالعات بازیابی‌شده به عنوان زمینه (Context) استفاده می کند‌تا پاسخ نهایی را تولید نماید.
 
 
معماری RA

In [246]:
print(chunks[0].metadata)

{'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-06-24T18:09:36+03:30', 'source': 'docs/rag_docs.pdf', 'file_path': 'docs/rag_docs.pdf', 'total_pages': 15, 'format': 'PDF 1.7', 'title': '', 'author': 'hamed safaei', 'subject': '', 'keywords': '', 'moddate': '2026-06-24T18:09:36+03:30', 'trapped': '', 'modDate': "D:20260624180936+03'30'", 'creationDate': "D:20260624180936+03'30'", 'page': 0}


In [273]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key=settings.OPENAI_API_KEY,
    base_url="https://api.gapgpt.app/v1"
)

In [251]:
from qdrant_client import QdrantClient

client = QdrantClient(
    url="http://localhost:6333"
)

d:\v3\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [274]:
from langchain_qdrant import QdrantVectorStore

vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url="http://localhost:6333",
    collection_name="rag_docs"
)

In [275]:
query_vector = embeddings.embed_query("معیار های ارزیابی rag")

In [276]:
results = client.query_points(
    collection_name="rag_docs",
    query=query_vector,
    limit=5,
)

In [277]:
for i, point in enumerate(results.points, start=1):
    print(f"\n===== Result {i} =====")
    print(f"Score: {point.score}")
    print(point.payload["page_content"])


===== Result 1 =====
Score: 0.65508956
۴
. معیارهای ارزیابی RAG
 
 ارزیابی کیفیت یک سیستم RAG از چندین جنبه مختلف انجام می‌شود.
 
۴.۱
 دقت بازیابی (Retrieval Accuracy)
 
این معیار نشان می‌دهد که آیا سیستم توانسته است قطعات مرتبط را با موفقیت بازیابی کند یا خیر.
 
شاخص‌های متداول در این حوزه عبارت اند از:
 
•
 
Precision@K: دقت در K نتیجه اول 
•
 
Recall@K: فراخوانی در K نتیجه اول 
•
 
MRR (Mean Reciprocal Rank): 
 میانگین رتبه معکوس 
•
 
NDCG (Normalized Discounted Cumulative Gain): معیار کیفیت رتبه‌بندی نتایج 
این شاخص‌ها میزان موفقیت سیستم در یافتن اطالعات مرتبط را اندازه‌گیری می‌کنند.
 
 
۴.۲
 کیفیت تولید (Generation Quality)
 
این معیار کیفیت پاسخ نهایی تولیدشده توسط مدل زبانی را ارزیابی می‌کند.
 
از جمله شاخص‌های رایج برای سنجش کیفیت تولید می توان به موارد زیر اشاره کرد:
 
•
 
BLEU
 
•
 
ROUGE
 
•
 
BERTScore

===== Result 2 =====
Score: 0.55078894
•
 تحلیل خودکار گزارش های سازمانی 
•
 پاسخ‌گویی به پرسش‌های مدیریتی بر اساس داده های داخلی 
•
 استخراج Insight از داده‌های ساخت‌یافته

In [365]:
"""
Parent-Child Retriever برای سند آموزش RAG فارسی

ساختار:
- Parent: هر عنوان اصلی (مقدمه، اجزای اصلی، چالش‌ها و ...)
- Child: هر زیرعنوان داخل آن عنوان

مرحله فعلی: استخراج و ساخت ساختار parent-child
مراحل بعدی (embedding, vectorstore) در جلسات آینده پیاده‌سازی می‌شوند.
"""

import re
from dataclasses import dataclass, field
from typing import Optional


# ─────────────────────────────────────────────
# ۱. دیتاکلاس‌ها
# ─────────────────────────────────────────────

@dataclass
class ChildChunk:
    """یک زیرعنوان با محتوای آن"""
    id: str                  # مثال: "2.1"
    title: str               # عنوان زیربخش
    content: str             # متن کامل زیربخش
    parent_id: str           # شناسه عنوان والد


@dataclass
class ParentChunk:
    """یک عنوان اصلی با تمام زیرعنوان‌هایش"""
    id: str                  # مثال: "2"
    title: str               # عنوان اصلی
    content: str             # متن کل عنوان (شامل همه زیرعنوان‌ها)
    children: list[ChildChunk] = field(default_factory=list)


# ─────────────────────────────────────────────
# ۲. داده خام سند (استخراج‌شده از PDF)
#    متن به همان ترتیب صفحات، نرمال‌سازی‌شده برای فارسی
# ─────────────────────────────────────────────

RAW_DOCUMENT = """
=== 1 === مقدمه و پیشینه
در دنیای امروز، مدل‌های زبانی بزرگ (LLM) به یکی از پایه‌های اصلی نرم‌افزارهای هوشمند تبدیل شده‌اند.
این مدل‌ها قادرند متون طبیعی را درک کرده، آن‌ها را تحلیل کنند و پاسخ‌هایی دقیق و روان تولید نمایند.
با این حال، یکی از بزرگ‌ترین چالش‌های این مدل‌ها آن است که دانش آن‌ها به زمان آموزش محدود می‌شود
و دسترسی مستقیمی به داده‌های جدید، اختصاصی یا سازمانی ندارند.

برای رفع این محدودیت، رویکرد «بازیابی-افزوده تولید» یا Retrieval-Augmented Generation (RAG)
مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:
۱. در مرحله اول، اطلاعات مرتبط از یک پایگاه دانش بیرونی بازیابی می‌شود.
۲. در مرحله دوم، مدل زبانی از این اطلاعات بازیابی‌شده به‌عنوان زمینه (Context) استفاده می‌کند
   تا پاسخ نهایی را تولید نماید.

معماری RAG نخستین بار توسط لوئیس و همکارانش در سال ۲۰۲۰ معرفی شد و از آن زمان تحولات
گسترده‌ای را تجربه کرده است. امروزه این معماری در طیف وسیعی از کاربردها، از جمله پرسش و پاسخ
سازمانی، چت‌بات‌های پشتیبانی مشتری، تحلیل اسناد حقوقی، سیستم‌های پزشکی و هوش تجاری (BI)
مورد استفاده قرار می‌گیرد.

=== 2 === اجزای اصلی یک سیستم RAG
یک سیستم RAG از چندین مؤلفه کلیدی تشکیل شده است که هر یک نقش مهمی در کیفیت نهایی پاسخ‌ها
ایفا می‌کنند.

--- 2.1 --- پردازش و بارگذاری اسناد (Document Ingestion)
اولین گام در ساخت یک سیستم RAG، آماده‌سازی و بارگذاری اسناد است. در این مرحله، متن خام از
اسناد استخراج شده، پاکسازی می‌شود و برای مراحل بعدی آماده می‌گردد.
اسناد می‌توانند در قالب‌های مختلفی مانند PDF، Word، HTML، متن ساده و حتی پایگاه‌های داده
ساختیافته ارائه شوند.

--- 2.2 --- تقسیم‌بندی متن (Chunking)
پس از بارگذاری اسناد، متن‌ها به قطعات کوچک‌تری به نام Chunk تقسیم می‌شوند. این کار به دلیل
محدودیت پنجره زمینه (Context Window) مدل‌های زبانی ضروری است.

استراتژی‌های متداول:
الف) تقسیم‌بندی با اندازه ثابت (Fixed-size Chunking): ساده و سریع، اما ممکن است انسجام معنایی را کاهش دهد.
ب) تقسیم‌بندی معنایی (Semantic Chunking): بر اساس مرزهای معنایی مانند پاراگراف‌ها؛ کیفیت بالاتر.
ج) تقسیم‌بندی با همپوشانی (Overlapping Chunks): چند توکن انتهایی هر قطعه در ابتدای قطعه بعدی تکرار می‌شود.

--- 2.3 --- تبدیل به بردار (Embedding)
مرحله بعدی، تبدیل هر قطعه متن به یک بردار عددی چندبعدی است. این بردارها نمایانگر معنای متن
در فضای برداری هستند. مدل‌های مشهور: text-embedding-ada-002، Sentence-BERT، multilingual-e5.
برای فارسی، مدل‌هایی مانند ParsBERT و مدل‌های بومی ایرانی پیشنهاد می‌شوند.

--- 2.4 --- پایگاه داده برداری (Vector Store)
الگوریتم‌های جستجو بر اساس شباهت کسینوسی یا فاصله اقلیدسی عمل می‌کنند. برای مقیاس‌های بزرگ
از الگوریتم‌هایی مانند HNSW و IVF برای جستجوی تقریبی (ANN) استفاده می‌شود.

--- 2.5 --- بازیابی و رتبه‌بندی (Retrieval & Reranking)
هنگامی که کاربر سؤالی مطرح می‌کند، ابتدا سؤال به بردار تبدیل شده، سپس نزدیک‌ترین قطعات
بازیابی می‌شوند (Top-K). مدل‌های رایج برای Reranking: Cross-Encoder، Cohere Rerank، Anthropic Rerank.

--- 2.6 --- تولید پاسخ (Generation)
سؤال کاربر به همراه قطعات بازیابی‌شده به‌عنوان Prompt به مدل زبانی ارسال می‌شود.
ساختار Prompt: ۱. System Prompt  ۲. Retrieved Context  ۳. سؤال کاربر

=== 3 === چالش‌های RAG برای زبان فارسی
پیاده‌سازی RAG برای زبان فارسی با چالش‌های خاصی همراه است.

--- 3.1 --- چندریختی متن فارسی
زبان فارسی دارای اشکال مختلف نوشتاری برای برخی کاراکترها است؛ مثلاً «ک» و «ك» یا «ی»، «ي» و «ئ».
این موضوع می‌تواند باعث عدم تطابق متون هنگام جستجو شود. راه‌حل: نرمال‌سازی متن (Text Normalization).

--- 3.2 --- اتصال کلمات و توکن‌بندی
فارسی یک زبان پیوندی (Agglutinative) است؛ پیشوندها و پسوندها به کلمات متصل می‌شوند.
مثال: «کتاب‌هایم» = کتاب + ها + ی + م
ابزارهای تخصصی: Hazm، NLTK Persian.

--- 3.3 --- کمبود داده‌های آموزشی
در مقایسه با انگلیسی، حجم داده‌های آموزشی فارسی بسیار کمتر است.
پروژه‌هایی مانند ParsBERT، FaBERT و mDeBERTa در حال کاهش این شکاف هستند.

--- 3.4 --- راست‌به‌چپ بودن متن
متن فارسی نیازمند پشتیبانی از نمایش RTL (Right-to-Left) است. باید تمام مراحل پردازش،
ذخیره‌سازی و نمایش از این ویژگی پشتیبانی کنند.

=== 4 === معیارهای ارزیابی RAG
ارزیابی کیفیت یک سیستم RAG از چندین جنبه انجام می‌شود.

--- 4.1 --- دقت بازیابی (Retrieval Accuracy)
شاخص‌های متداول: Precision@K، Recall@K، MRR (Mean Reciprocal Rank)، NDCG.

--- 4.2 --- کیفیت تولید (Generation Quality)
شاخص‌های رایج: BLEU، ROUGE، BERTScore.
چارچوب RAGAS معیارهایی مانند Faithfulness، Answer Relevancy، Context Precision را اندازه‌گیری می‌کند.

--- 4.3 --- صحت استناد (Faithfulness)
مدل نباید دچار توهم‌زایی (Hallucination) شود؛ یعنی اطلاعاتی ارائه دهد که در اسناد بازیابی‌شده
وجود ندارند. این معیار در حوزه‌های پزشکی، حقوقی و مالی اهمیت بسیار زیادی دارد.

=== 5 === معماری‌های پیشرفته RAG

--- 5.1 --- HyDE و روش‌های پیشرفته RAG
در HyDE (Hypothetical Document Embeddings)، ابتدا یک سند فرضی بر اساس پرسش تولید می‌شود
و سپس برای جستجو استفاده می‌گردد. تکنیک‌های دیگر: Query Rewriting، Multi-hop Retrieval،
Fusion of Retrieval Results.

--- 5.2 --- Agentic RAG
یک Agent تصمیم می‌گیرد آیا بازیابی لازم است، از کدام منابع استفاده شود و چند بار تکرار شود.
ابزارهای پیاده‌سازی: LangGraph، CrewAI.

--- 5.3 --- GraphRAG
به جای قطعات متن، یک Knowledge Graph از اسناد ساخته می‌شود. موجودیت‌ها و روابط شناسایی
و ذخیره می‌شوند. برای پرسش‌هایی که نیاز به ترکیب چندین منبع دارند بسیار مؤثر است.

=== 6 === پیاده‌سازی عملی با LangChain
LangChain یکی از پرکاربردترین چارچوب‌ها برای ساخت سیستم‌های RAG است.
مرحله ۱: بارگذاری اسناد با PyPDFLoader یا DirectoryLoader.
مرحله ۲: تقسیم‌بندی با RecursiveCharacterTextSplitter (تنظیم chunk_size و chunk_overlap برای فارسی).
مرحله ۳: تبدیل به بردار و ذخیره در Chroma یا FAISS.
مرحله ۴: ساخت زنجیره RAG با RetrievalQA یا LCEL.

=== 7 === کاربرد RAG در هوش تجاری (BI)
RAG در حوزه هوش تجاری کاربردهای مهمی دارد:
- تحلیل خودکار گزارش‌های سازمانی
- پاسخگویی به پرسش‌های مدیریتی بر اساس داده‌های داخلی
- تولید داشبوردهای هوشمند با توضیحات متنی
- استخراج بینش (Insight) از داده‌های ساختیافته و غیرساختیافته

=== 8 === بهترین رویکردها و توصیه‌ها
- تمرکز جدی بر نرمال‌سازی متن فارسی با Hazm
- آزمایش مدل‌های مختلف Embedding (256-512 و 512-1024 توکن)
- تنظیم دقیق chunk size بر اساس نوع محتوا
- استفاده از Reranking برای پرسش‌های پیچیده (Cross-Encoder، Cohere، Anthropic)
- طراحی سیستم ارزیابی مستمر با Golden QA Pairs
- استفاده از ابزارهای Observability مانند LangSmith یا Langfuse

=== 9 === آینده RAG
- Streaming RAG: تولید پاسخ به‌صورت تدریجی
- Multimodal RAG: پردازش همزمان متن، تصویر، جدول و نمودار
- RAG با حافظه بلندمدت: نگهداری تاریخچه مکالمات
با افزایش ظرفیت مدل‌ها و گسترش Context Window، برخی کاربردهای سنتی RAG تغییر خواهند کرد،
اما در سیستم‌هایی با داده‌های بزرگ همچنان نقش کلیدی خواهد داشت.

=== 10 === نتیجه‌گیری
RAG یک معماری قدرتمند است که شکاف بین توانایی‌های LLM و نیازهای واقعی سازمان‌ها را پر می‌کند.
با رعایت اصول مربوط به زبان فارسی، انتخاب مدل‌های مناسب، طراحی درست pipeline و ارزیابی مستمر،
می‌توان سیستم‌های RAG با کیفیت بالا برای کاربران فارسی‌زبان توسعه داد.
"""


# ─────────────────────────────────────────────
# ۳. پارسر ساختار parent-child
# ─────────────────────────────────────────────

def parse_document(raw_text: str) -> list[ParentChunk]:
    """
    متن سند را خوانده و ساختار parent-child می‌سازد.

    قراردادها در متن خام:
      === N === عنوان    →  Parent
      --- N.M --- عنوان →  Child
    """
    lines = raw_text.strip().splitlines()

    parents: list[ParentChunk] = []
    current_parent: Optional[ParentChunk] = None
    current_child: Optional[ChildChunk] = None
    current_lines: list[str] = []

    def flush_child():
        """محتوای جاری را در child فعلی ذخیره می‌کند."""
        nonlocal current_child, current_lines
        if current_child is not None:
            current_child.content = "\n".join(current_lines).strip()
            current_parent.children.append(current_child)
            current_child = None
            current_lines = []

    def flush_parent_body():
        """محتوای بین عنوان parent و اولین child را در content والد می‌گذارد."""
        nonlocal current_lines
        if current_parent is not None and not current_parent.children and current_lines:
            current_parent.content = "\n".join(current_lines).strip()
            current_lines = []

    for line in lines:
        line = line.strip()

        # ── parent header ──
        parent_match = re.match(r"^===\s*([\d]+)\s*===\s*(.+)$", line)
        if parent_match:
            flush_child()
            flush_parent_body()
            if current_parent:
                # محتوای کامل parent = content خودش + همه children
                _rebuild_parent_content(current_parent)
                parents.append(current_parent)

            pid = parent_match.group(1).strip()
            ptitle = parent_match.group(2).strip()
            current_parent = ParentChunk(id=pid, title=ptitle, content="")
            current_lines = []
            continue

        # ── child header ──
        child_match = re.match(r"^---\s*([\d]+\.[\d]+)\s*---\s*(.+)$", line)
        if child_match:
            flush_child()
            flush_parent_body()
            cid = child_match.group(1).strip()
            ctitle = child_match.group(2).strip()
            current_child = ChildChunk(
                id=cid,
                title=ctitle,
                content="",
                parent_id=current_parent.id if current_parent else "",
            )
            current_lines = []
            continue

        # ── خط معمولی ──
        current_lines.append(line)

    # flush آخرین‌ها
    flush_child()
    flush_parent_body()
    if current_parent:
        _rebuild_parent_content(current_parent)
        parents.append(current_parent)

    return parents


def _rebuild_parent_content(parent: ParentChunk):
    """
    محتوای کامل parent را از content مستقیم + محتوای همه children می‌سازد.
    این همان چیزی است که در retrieval به‌عنوان context بزرگ‌تر برگردانده می‌شود.
    """
    parts = []
    if parent.content:
        parts.append(parent.content)
    for child in parent.children:
        parts.append(f"[{child.id}] {child.title}\n{child.content}")
    parent.content = "\n\n".join(parts)


# ─────────────────────────────────────────────
# ۴. ساختار Parent-Child Retriever (بدون embedding)
# ─────────────────────────────────────────────

class ParentChildRetriever:
    """
    نگه‌دارنده ساختار parent-child و رابط جستجوی ساده متنی.

    در مرحله بعدی:
    - children به vectorstore اضافه می‌شوند (embedding می‌شوند).
    - parents فقط در docstore نگه‌داری می‌شوند (بدون embedding).
    - جستجو روی child انجام می‌شود، اما parent کامل برگردانده می‌شود.
    """

    def __init__(self, parents: list[ParentChunk]):
        self.parents: dict[str, ParentChunk] = {p.id: p for p in parents}
        # ایندکس معکوس: child_id → parent_id
        self.child_to_parent: dict[str, str] = {}
        self.children: dict[str, ChildChunk] = {}

        for parent in parents:
            for child in parent.children:
                self.children[child.id] = child
                self.child_to_parent[child.id] = parent.id

    # ── جستجوی ساده متنی (placeholder تا embedding اضافه شود) ──

    def search_children(self, query: str, top_k: int = 3) -> list[ChildChunk]:
        """جستجوی keyword ساده در عنوان و محتوای children."""
        query_lower = query.lower()
        results = []
        for child in self.children.values():
            score = 0
            if query_lower in child.title.lower():
                score += 2
            if query_lower in child.content.lower():
                score += 1
            if score > 0:
                results.append((score, child))
        results.sort(key=lambda x: x[0], reverse=True)
        return [c for _, c in results[:top_k]]

    def get_parent_for_child(self, child_id: str) -> Optional[ParentChunk]:
        """با داشتن child_id، parent کامل را برمی‌گرداند."""
        parent_id = self.child_to_parent.get(child_id)
        return self.parents.get(parent_id) if parent_id else None

    def retrieve(self, query: str, top_k: int = 3) -> list[dict]:
        """
        جستجو در children، سپس parent کامل را برمی‌گرداند.
        این الگوی اصلی Parent-Child Retrieval است.
        """
        matched_children = self.search_children(query, top_k)
        results = []
        seen_parents = set()
        for child in matched_children:
            parent = self.get_parent_for_child(child.id)
            results.append({
                "matched_child": child,
                "parent": parent,
                "already_seen": parent.id in seen_parents if parent else True,
            })
            if parent:
                seen_parents.add(parent.id)
        return results


# ─────────────────────────────────────────────
# ۵. اجرا و نمایش ساختار
# ─────────────────────────────────────────────

def print_structure(parents: list[ParentChunk]):
    print("=" * 60)
    print("ساختار Parent-Child سند RAG فارسی")
    print("=" * 60)
    for parent in parents:
        preview = parent.content[:80].replace("\n", " ") + "..."
        print(f"\n📁 Parent [{parent.id}]: {parent.title}")
        print(f"   محتوا (پیش‌نمایش): {preview}")
        if parent.children:
            for child in parent.children:
                child_preview = child.content[:60].replace("\n", " ") + "..."
                print(f"   └─ Child [{child.id}]: {child.title}")
                print(f"             محتوا: {child_preview}")
        else:
            print("   └─ (بدون زیرعنوان)")
    print("\n" + "=" * 60)
    total_children = sum(len(p.children) for p in parents)
    print(f"✅ تعداد کل Parents: {len(parents)}")
    print(f"✅ تعداد کل Children: {total_children}")
    print("=" * 60)


def demo_retrieval(retriever: ParentChildRetriever):
    print("\n--- نمونه Retrieval ---")
    test_queries = ["Embedding", "فارسی", "ارزیابی"]
    for q in test_queries:
        print(f"\n🔍 پرسش: '{q}'")
        results = retriever.retrieve(q, top_k=2)
        if not results:
            print("  نتیجه‌ای یافت نشد.")
        for r in results:
            child = r["matched_child"]
            parent = r["parent"]
            print(f"  ↳ Child یافت‌شده: [{child.id}] {child.title}")
            if parent:
                print(f"     Parent برگشتی: [{parent.id}] {parent.title}")


if __name__ == "__main__":
    # پارس سند
    parents = parse_document(RAW_DOCUMENT)

    # نمایش ساختار
    print_structure(parents)

    # ساخت retriever
    retriever = ParentChildRetriever(parents)

    # نمونه retrieval
    demo_retrieval(retriever)

    print("\n✅ ساختار parent-child آماده است.")
    print("📌 مرحله بعدی: اضافه کردن embedding و vectorstore.")

ساختار Parent-Child سند RAG فارسی

📁 Parent [1]: مقدمه و پیشینه
   محتوا (پیش‌نمایش): در دنیای امروز، مدل‌های زبانی بزرگ (LLM) به یکی از پایه‌های اصلی نرم‌افزارهای هو...
   └─ (بدون زیرعنوان)

📁 Parent [2]: اجزای اصلی یک سیستم RAG
   محتوا (پیش‌نمایش): یک سیستم RAG از چندین مؤلفه کلیدی تشکیل شده است که هر یک نقش مهمی در کیفیت نهایی...
   └─ Child [2.1]: پردازش و بارگذاری اسناد (Document Ingestion)
             محتوا: اولین گام در ساخت یک سیستم RAG، آماده‌سازی و بارگذاری اسناد ...
   └─ Child [2.2]: تقسیم‌بندی متن (Chunking)
             محتوا: پس از بارگذاری اسناد، متن‌ها به قطعات کوچک‌تری به نام Chunk ...
   └─ Child [2.3]: تبدیل به بردار (Embedding)
             محتوا: مرحله بعدی، تبدیل هر قطعه متن به یک بردار عددی چندبعدی است. ...
   └─ Child [2.4]: پایگاه داده برداری (Vector Store)
             محتوا: الگوریتم‌های جستجو بر اساس شباهت کسینوسی یا فاصله اقلیدسی عم...
   └─ Child [2.5]: بازیابی و رتبه‌بندی (Retrieval & Reranking)
             محتوا: هنگامی که کاربر سؤالی مطرح می‌کند، ا